In [ ]:
from pathlib import Path

from typing import cast
import pickle as pkl

import matplotlib.pyplot as plt
import numpy.typing as npt
import numpy as np

import networkx as nx

from topostats.classes import TopoStats
from topostats.plottingfuncs import Colormap

from skimage.measure import label

from topostats.utils import convolve_skeleton

CMAP = Colormap().get_cmap()
VMIN = -3
VMAX = 4

In [ ]:
def plot(image: npt.NDArray[np.uint8]) -> None:
    plt.imshow(image, cmap=CMAP, vmin=VMIN, vmax=VMAX)
    plt.show()


def create_mask_from_points(image_shape: tuple[int, int], points: list[tuple[int, int]]) -> npt.NDArray[np.uint8]:
    mask = np.zeros(image_shape, dtype=np.uint8)
    for point in points:
        mask[point[0], point[1]] = 1
    return mask


def plot_mask_of_points(image_shape: tuple[int, int], points: list[tuple[int, int]]) -> None:
    mask = create_mask_from_points(image_shape, points)
    plt.imshow(mask, cmap="gray")
    plt.show()

In [ ]:
with open(Path("../debug_topostats.pkl"), "rb") as f:
    topostats = cast(TopoStats, pkl.load(f))
assert isinstance(topostats, TopoStats), "Loaded object is not a TopoStats instance"
assert topostats is not None

print(topostats)
print(topostats.grain_crops)
grain_crops = topostats.require_grain_crops()
grain_crop = grain_crops[2]
plot(grain_crop.image)
nodes = grain_crop.nodes
print(nodes.keys())
node_0 = nodes[0]
node_1 = nodes[1]
print(node_0)
print(node_0.branch_stats.keys())
print(node_0.unmatched_branch_stats.keys())


plt.imshow(grain_crop.image)
plt.imshow(node_0.node_branch_mask, alpha=0.5)
print("=== branch stats ===")
for branch_id, branch_stat in node_0.branch_stats.items():
    print(f"---branch id {branch_id}:\n{branch_stat}")
    random_color = np.random.rand(
        3,
    )
    for coord in branch_stat.ordered_coords:
        plt.plot(coord[1], coord[0], color=random_color, marker="o", markersize=2)

for coord in node_0.node_coords:
    plt.plot(coord[1], coord[0], "ro", markersize=3)
plt.show()

# Create a skeleton image without the nodes - just branches
grain_branches_image = grain_crop.skeleton
plt.imshow(grain_branches_image)
plt.show()
for node_index, node in nodes.items():
    print(f"=== node index {node_index} ===")
    node_coords = node.node_coords
    assert node_coords is not None, "Node coordinates are None"
    for node_coord in node_coords:
        grain_branches_image[node_coord[0], node_coord[1]] = -1
plt.imshow(grain_branches_image)
plt.show()

# Label the branches
grain_branches_image_without_nodes = np.where(grain_branches_image == -1, 0, grain_branches_image)
labelled_grain_branches_image = label(grain_branches_image_without_nodes)
# add the node pixels back to the labelled image
labelled_branches_image = np.where(grain_branches_image == -1, -1, labelled_grain_branches_image)
plt.imshow(labelled_branches_image)
plt.show()


# Associate the skeleton branches with the node branches
branch_node_connections: dict[int, tuple[int | None, int | None]] = {}  # branch_id -> (node_index_1, node_index_2)
# For each branch, look at each end and see what node it is touching, then associate that branch with that node
for branch_id in np.unique(labelled_branches_image):
    if branch_id == 0 or branch_id == -1:  # Skip background and node pixels
        continue
    # Find the endpoints of the branch by convolving the branch
    branch_mask = np.where(labelled_branches_image == branch_id, 1, 0)
    convolved_branch = convolve_skeleton(branch_mask)
    plt.imshow(convolved_branch)
    plt.show()
    endpoints = np.argwhere(convolved_branch == 2)
    assert len(endpoints) == 2, f"Branch {branch_id} does not have exactly 2 endpoints: {endpoints}"
    # Find which nodes the endpoints are touching
    node_indices = []
    for endpoint in endpoints:
        node_index = None
        for node_index, node in nodes.items():
            node_confidence = node.confidence
            assert node_confidence is not None, "Node confidence is None"
            assert node.node_coords is not None, "Node coordinates are None"
            neighbour_pixels = [(endpoint[0] + dy, endpoint[1] + dx) for dy in [-1, 0, 1] for dx in [-1, 0, 1]]
            if any(neighbour_pixel in node.node_coords for neighbour_pixel in neighbour_pixels):
                node_indices.append(node_index)
                break
    assert len(node_indices) == 2, f"Branch {branch_id} does not have exactly 2 associated nodes: {node_indices}"
    # Add the branch and its associated nodes to the dictionary
    branch_node_connections[int(branch_id)] = (node_indices[0], node_indices[1])

print(branch_node_connections)

# Construct a suggestions for each node for which branches to pair together
branch_connection_suggestions: dict[int, list[tuple[int, int, float]]] = (
    {}
)  # node_index -> (branch_index_1, branch_index_2, confidence)
for node_index, node in nodes.items():
    node_pairing_suggestions: list[tuple[int, int, float]] = []
    node_confidence = node.confidence
    assert node_confidence is not None
    node_branch_stats = node.branch_stats
    assert node_branch_stats is not None
    for node_branch_pair_data in node_branch_stats.values():
        branch_pair_coords = node_branch_pair_data.ordered_coords
        assert branch_pair_coords is not None
        # Get the indexes of the branches in the pair
        branch_pair_ids = set()
        for branch_pair_coord in branch_pair_coords:
            branch_id = labelled_branches_image[branch_pair_coord[0], branch_pair_coord[1]]
            assert branch_id != 0, f"Branch pair coordinate {branch_pair_coord} is not part of a branch"
            branch_pair_ids.add(int(branch_id))
        # Remove any -1s since we don't care about the node here
        branch_pair_ids.discard(-1)
        assert (
            len(branch_pair_ids) == 2
        ), f"Branch pair coordinates {branch_pair_coords} do not correspond to exactly 2 branches: {branch_pair_ids}"
        node_pairing_suggestions.append((branch_pair_ids.pop(), branch_pair_ids.pop(), np.round(node_confidence, 3)))
    branch_connection_suggestions[node_index] = node_pairing_suggestions

print(branch_connection_suggestions)

In [ ]:
from typing import Generator
import itertools


def get_permutations_of_options_with_costs(
    dict_of_items_with_options_and_costs: dict[int, list[tuple[int, float]]],
) -> list[tuple[dict[int, tuple[int, float]], float]]:
    """
    Get costs of all permutations of options for a dictionary of items with varying options.

    Given a dictionary of items with their representative options and costs, return all permutations of the options
    with their associated total costs.

    Parameters
    ----------
    dict_of_items_with_options_and_costs : dict[int, list[tuple[int, float]]]
        A dictionary where keys are item identifiers and values are lists of tuples, each containing an option and its
        cost.


    Returns
    -------
    list[tuple[dict[int, tuple[int, float]], float]]
        A list of tuples, where each tuple contains a dictionary mapping item identifiers to their selected option and
        the total cost of that permutation.
    """
    # Unpack the items (ID, options) to compute all combinations
    keys, option_lists = zip(*dict_of_items_with_options_and_costs.items())
    all_permutations = []
    for combination in itertools.product(*option_lists):
        permutation_dict = dict(zip(keys, combination))
        total_cost = sum(cost for _, cost in combination)
        all_permutations.append((permutation_dict, total_cost))
    return all_permutations


def get_all_pairing_lists_for_items(items: list[int]) -> Generator[list[tuple[int, int]], None, None]:
    """
    Generates all possible pairing lists for a given list of items.

    Parameters
    ----------
    items : list[int]
        A list of items to be paired.

    Yields
    ------
    list[tuple[int, int]]
        A list of tuples, where each tuple contains a pair of items.
    """
    # Base case: no items left
    if not items:
        yield []
        return

    # Choose the current item to pair with the rest
    first = items[0]
    rest = items[1:]

    # Pair this item with each of the remaining items
    for i, partner in enumerate(rest):
        pair = (first, partner)
        remaining_items = rest[:i] + rest[i + 1 :]

        # Recursively pair the remaining elements that we don't pair with the current item
        for sub_pairing in get_all_pairing_lists_for_items(remaining_items):
            yield [pair] + sub_pairing


def create_vertex_edge_to_edge_map(
    vertex_suggestions: dict[int, list[tuple[int, int]]],
) -> dict[tuple[int, int], int]:
    """Create a mapping of vertex -> in_edge, out_edge based on the suggestions for which branches to pair."""
    # Create a mapping of vertex -> in_edge, out_edge
    vertex_edge_to_edge_map: dict[tuple[int, int], int] = {}  # (vertex_index, in_edge_index) -> out_edge_index
    for vertex_index, suggestions in vertex_suggestions.items():
        for in_edge_index, out_edge_index in suggestions:
            vertex_edge_to_edge_map[(vertex_index, in_edge_index)] = out_edge_index
            vertex_edge_to_edge_map[(vertex_index, out_edge_index)] = in_edge_index

    return vertex_edge_to_edge_map


def get_cycles_from_graph(
    edges_to_vertices: dict[int, tuple[int, int]], vertex_suggestions: dict[int, list[tuple[int, int]]]
) -> list[list[tuple[int, int]]]:
    """
    Get cycles from a 2k-regular graph, using suggestions for which branches to pair.
    """
    vertex_edge_to_edge_map = create_vertex_edge_to_edge_map(vertex_suggestions)

    # Construct cycles based on the vertex_edge_to_edge_map
    cycles: list[list[tuple[int, int]]] = []  # Each cycle is a list of (edge_index, vertex_index) tuples
    current_cycle: list[tuple[int, int]] = []
    # Start at a vertex, choose an edge and follow it
    start_vertex = next(iter(vertex_suggestions.keys()))
    current_vertex = start_vertex
    current_vertex_suggestions = vertex_suggestions[current_vertex]
    current_vertex_suggestion = current_vertex_suggestions[0]
    current_edge = current_vertex_suggestion[0]
    visited_edges = set()
    while True:
        # Add the current edge to the cycle
        current_cycle.append((current_edge, current_vertex))
        # Get the next edge to follow
        next_edge = vertex_edge_to_edge_map.get((current_vertex, current_edge))
        # Get the next vertex to move to
        edge_vertices = edges_to_vertices[current_edge]
        # Avoid staying on the same vertex
        next_vertex = edge_vertices[0] if edge_vertices[1] == current_vertex else edge_vertices[1]
        # Check if we've visited this edge before
        if next_edge in visited_edges:
            # Completed a cycle
            cycles.append(current_cycle)
            # Start a new cycle if there are unvisited edges
            unvisited_edges = set(edges_to_vertices.keys()) - visited_edges
            if not unvisited_edges:
                break  # All edges visited, exit the loop
            # Start a new cycle with an unvisited edge
            current_edge = unvisited_edges.pop()
            current_vertex = edges_to_vertices[current_edge][0]  # Start from one of the vertices of the new edge
            current_cycle = []
            continue
        else:
            # Not completed a cycle, mark the current edge as visited and move to the next vertex
            visited_edges.add(current_edge)
            current_edge = next_edge
            current_vertex = next_vertex

    assert len(visited_edges) == len(edges_to_vertices), "Not all edges were visited in the traversal"

    return cycles


def create_single_trace(
    edges_to_vertices: dict[int, tuple[int, int]], vertex_suggestions_with_costs: dict[int, list[tuple[int, int, float]]]
) -> list[tuple[int, int]]:
    """
    Create a single Eulerian trace from a 2k-regular graph, using suggestions for which branches to pair.

    Parameters
    ----------
    edges_to_vertices : dict
        A dictionary mapping edge indexes to two vertex indexes.
    vertex_suggestions : dict
        A dictionary mapping vertex indexes to a list of tuples, each containing two branch indexes and a
        confidence score.

    Returns
    -------
    list[tuple[int, int]]
        A list of edge to vertex connections representing a single Eulerian trace.
    """
    # For each vertex, get the permutations of the suggestions with associated costs
    vertex_permutations_with_costs: dict[int, list[tuple[list[tuple[int, int]], float]]] = {}
    # Dict of vertex_index -> list of pairing options with the total cost of that pairing option
    for vertex_index, suggestions in vertex_suggestions_with_costs.items():
        all_edge_indexes = set()
        for in_edge_index, out_edge_index, confidence in suggestions:
            all_edge_indexes.add(in_edge_index)
            all_edge_indexes.add(out_edge_index)
        # Get all permutations of the possible connections for this vertex
        all_pairing_options = list(get_all_pairing_lists_for_items(list(all_edge_indexes)))
        # Associate costs with these pairings
        all_pairing_options_with_costs: list[tuple[list[tuple[int, int]], float]] = (
            []
        )  # list of (pairing_option, cost) tuples
        for pairing_option in all_pairing_options:
            cost = 0
            for in_edge_index, out_edge_index, confidence in suggestions:
                if (in_edge_index, out_edge_index) not in pairing_option and (
                    out_edge_index,
                    in_edge_index,
                ) not in pairing_option:
                    cost += confidence  # Add the confidence score to the cost if this suggestion is not used
            all_pairing_options_with_costs.append((pairing_option, cost))
        vertex_permutations_with_costs[vertex_index] = all_pairing_options_with_costs

    # Create a mapping between vertex index and the pairing options with costs for that vertex - just
    # for simplicity with handling the itertools product later on
    vertex_pairing_options_with_costs: dict[int, list[tuple[int, float]]] = {}
    for vertex_index, pairing_options_with_costs in vertex_permutations_with_costs.items():
        vertex_pairing_options_with_costs[vertex_index] = []
        # For each pairing option with cost, add that
        for pairing_option_index, (_pairing_option, cost) in enumerate(pairing_options_with_costs):
            vertex_pairing_options_with_costs[vertex_index].append((pairing_option_index, cost))

    # Find the combination of pairing options across all vertices and their costs
    global_all_vertex_pairing_options_with_costs = get_permutations_of_options_with_costs(
        vertex_pairing_options_with_costs
    )

    # Sort the global pairing options by their total cost in ascending order
    global_all_vertex_pairing_options_with_costs.sort(key=lambda x: x[1])

    # Iterate through the sorted global pairing options and find the first one that results in a single Eulerian trace
    for global_pairing_option, total_cost in global_all_vertex_pairing_options_with_costs:
        # Construct the vertex_suggestions based on the global pairing option
        vertex_suggestions_for_this_option: dict[int, list[tuple[int, int]]] = {}
        for vertex_index, pairing_option_index in global_pairing_option.items():
            LEFT OFF HERE
            

    raise ValueError(
        "Too many attempts to find a single Eulerian trace, giving up. This should not happen. Look into this"
    )

In [ ]:
create_single_trace(branch_node_connections, branch_connection_suggestions)

In [ ]:
from typing import Generator


def get_all_pairing_lists_for_items(items: list[int]) -> Generator[list[tuple[int, int]], None, None]:
    """
    Generates all possible pairing lists for a given list of items.

    Parameters
    ----------
    items : list[int]
        A list of items to be paired.

    Yields
    ------
    list[tuple[int, int]]
        A list of tuples, where each tuple contains a pair of items.
    """
    # Base case: no items left
    if not items:
        yield []
        return

    # Choose the current item to pair with the rest
    first = items[0]
    rest = items[1:]

    # Pair this item with each of the remaining items
    for i, partner in enumerate(rest):
        pair = (first, partner)
        remaining_items = rest[:i] + rest[i + 1 :]

        # Recursively pair the remaining elements that we don't pair with the current item
        for sub_pairing in get_all_pairing_lists_for_items(remaining_items):
            yield [pair] + sub_pairing


# Example with 6 items
items = ["A", "B", "C", "D", "E", "F"]
pairings = list(get_all_pairing_lists_for_items(items))

print(f"Total pairings for {len(items)} items: {len(pairings)}")
for pair in pairings:
    print(pair)

absent_pair_costs = {
    ("A", "B"): 0.1,
    ("C", "D"): 0.2,
    ("E", "F"): 0.3,
}

# Add the costs if the pair is not present in the list
pairings_with_costs = []
for pairing_list in pairings:
    cost = 0
    for absent_pair, cost_value in absent_pair_costs.items():
        if absent_pair not in pairing_list:
            cost += cost_value
    pairings_with_costs.append((pairing_list, cost))
print(f"Total pairings with costs for {len(items)} items: {len(pairings_with_costs)}")
for p, c in pairings_with_costs:
    print(f"Pairing: {p}, Cost: {c}")

In [ ]:
import itertools

# Key-value mapping of list IDs to their respective options
vertex_options_and_costs: dict[int, list[tuple[int, float]]] = {
    0: [(0, 0.0), (1, 0.2)],
    1: [(0, 0.0), (1, 0.4)],
}

# Unpack the items (ID, options) to compute all combinations
keys, option_lists = zip(*vertex_options_and_costs.items())
all_vertex_pairing_options = [dict(zip(keys, combination)) for combination in itertools.product(*option_lists)]

print(f"Total vertex pairing options: {len(all_vertex_pairing_options)}")
for option in all_vertex_pairing_options:
    print(option)


def get_permutations_of_options_with_costs(
    dict_of_items_with_options_and_costs: dict[int, list[tuple[int, float]]],
) -> list[tuple[dict[int, tuple[int, float]], float]]:
    """
    Get costs of all permutations of options for a dictionary of items with varying options.

    Given a dictionary of items with their representative options and costs, return all permutations of the options
    with their associated total costs.

    Parameters
    ----------
    dict_of_items_with_options_and_costs : dict[int, list[tuple[int, float]]]
        A dictionary where keys are item identifiers and values are lists of tuples, each containing an option and its
        cost.


    Returns
    -------
    list[tuple[dict[int, tuple[int, float]], float]]
        A list of tuples, where each tuple contains a dictionary mapping item identifiers to their selected option and
        the total cost of that permutation.
    """
    # Unpack the items (ID, options) to compute all combinations
    keys, option_lists = zip(*dict_of_items_with_options_and_costs.items())
    all_permutations = []
    for combination in itertools.product(*option_lists):
        permutation_dict = dict(zip(keys, combination))
        total_cost = sum(cost for _, cost in combination)
        all_permutations.append((permutation_dict, total_cost))
    return all_permutations


get_permutations_of_options_with_costs(vertex_options_and_costs)